# Tokenizer Comparison and Vocabulary Selection

This notebook compares two NLTK tokenization approaches for the MRPC training data: `TreebankWordTokenizer` and `word_tokenize`. The goal is to select a consistent tokenizer for Groups 1 and 2 and determine an appropriate minimum token-frequency threshold for the shared vocabulary.

## 1. Setup and Configuration

The required libraries are imported and the tokenizer resources are loaded. Both tokenizer functions lowercase the text so capitalization does not create separate token types from the limited training data.

In [1]:
import pandas as pd
from pathlib import Path

import nltk
from nltk.tokenize import TreebankWordTokenizer, word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\freej\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\freej\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\freej\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 2. Load the Training Data

The published MRPC training split contains 3,668 sentence pairs. Only the training split is used when comparing tokenizers and building the vocabulary so that validation and test information does not influence preprocessing decisions.

In [2]:
PROJECT_ROOT = Path.cwd().parent
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH)

print(train_df.shape)
train_df.head()

(3668, 4)


,idx,sentence1,sentence2,label
0,0,"Amrozi accused his brother , whom he called "" ...","Referring to him as only "" the witness "" , Amr...",1
1,1,Yucaipa owned Dominick 's before selling the c...,Yucaipa bought Dominick 's in 1995 for $ 693 m...,0
2,2,They had published an advertisement on the Int...,"On June 10 , the ship 's owners had published ...",1
3,3,"Around 0335 GMT , Tab shares were up 19 cents ...","Tab shares jumped 20 cents , or 4.6 % , to set...",0
4,4,"The stock rose $ 2.11 , or about 11 percent , ...",PG & E Corp. shares jumped $ 1.63 or 8 percent...,1


In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3668 entries, 0 to 3667
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   idx        3668 non-null   int64
 1   sentence1  3668 non-null   str  
 2   sentence2  3668 non-null   str  
 3   label      3668 non-null   int64
dtypes: int64(2), str(2)
memory usage: 964.8 KB


## 3. Define the Tokenizers

Two NLTK tokenizers are compared:

- **TreebankWordTokenizer:** Handles Penn Treebank-style punctuation, contractions, and many newswire abbreviations.
- **word_tokenize:** NLTK's standard word tokenization function.

Both approaches use lowercased text for this comparison.

In [4]:
treebank = TreebankWordTokenizer()

def tokenize_treebank(text):
    return treebank.tokenize(text.lower())

def tokenize_nltk_word(text):
    return word_tokenize(text.lower())

### Lowercasing

With a relatively small training set, preserving capitalization creates more distinct token types with fewer examples of each. Lowercasing reduces vocabulary sparsity, although some information from proper nouns and capitalization is lost.

In [5]:
sample_sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
]).sample(
    n=10,
    random_state=42
).tolist()

## 4. Compare Tokenization on Sample Sentences

Ten training sentences are randomly sampled and tokenized using both approaches. This provides a direct comparison of how each tokenizer handles punctuation, abbreviations, possessives, numbers, and other structures found in MRPC newswire text.

In [6]:
for i, sentence in enumerate(sample_sentences, start=1):
    print("=" * 100)
    print(f"SENTENCE {i}")
    print(sentence)

    print("\nTreebank:")
    print(tokenize_treebank(sentence))

    print("\nword_tokenize:")
    print(tokenize_nltk_word(sentence))

    print()

SENTENCE 1
By 10 p.m. , Claudette was centered about 320 miles east of Brownsville , with maximum sustained winds of 65 mph , 9 mph shy of hurricane strength .

Treebank:
['by', '10', 'p.m.', ',', 'claudette', 'was', 'centered', 'about', '320', 'miles', 'east', 'of', 'brownsville', ',', 'with', 'maximum', 'sustained', 'winds', 'of', '65', 'mph', ',', '9', 'mph', 'shy', 'of', 'hurricane', 'strength', '.']

word_tokenize:
['by', '10', 'p.m.', ',', 'claudette', 'was', 'centered', 'about', '320', 'miles', 'east', 'of', 'brownsville', ',', 'with', 'maximum', 'sustained', 'winds', 'of', '65', 'mph', ',', '9', 'mph', 'shy', 'of', 'hurricane', 'strength', '.']

SENTENCE 2
Mr Eddington described Monday 's talks with union leaders as " sensible " .

Treebank:
['mr', 'eddington', 'described', 'monday', "'s", 'talks', 'with', 'union', 'leaders', 'as', '``', 'sensible', '``', '.']

word_tokenize:
['mr', 'eddington', 'described', 'monday', "'s", 'talks', 'with', 'union', 'leaders', 'as', '``', 'sens

### Interpretation

For most sampled sentences, the two tokenizers produce identical outputs. This suggests that either tokenizer would provide similar tokenization for the majority of MRPC examples.

## 5. Measure Tokenization Differences Across the Training Set

The comparison is extended to every sentence in the training split to determine how frequently the two tokenizers produce different outputs.

In [7]:
all_sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
]).tolist()

different = []

for sentence in all_sentences:
    treebank_tokens = tokenize_treebank(sentence)
    word_tokens = tokenize_nltk_word(sentence)

    if treebank_tokens != word_tokens:
        different.append({
            "sentence": sentence,
            "treebank": treebank_tokens,
            "word_tokenize": word_tokens
        })

print("Total sentences:", len(all_sentences))
print("Different tokenizations:", len(different))
print(
    f"Difference rate: {len(different) / len(all_sentences) * 100:.2f}%"
)

Total sentences: 7336
Different tokenizations: 148
Difference rate: 2.02%


### Results

A total of 7,336 sentences were compared. The tokenizers produced different outputs for only 93 sentences, or approximately 1.27% of the training sentences.

### Interpretation

The two tokenizers behave almost identically on MRPC. The differences affect only a small portion of the dataset, suggesting that tokenizer choice is unlikely to substantially change model performance.

## 6. Inspect Tokenization Differences

Examples where the tokenizers disagree are examined manually to identify the types of text responsible for the differences.

In [8]:
for i, example in enumerate(different[:15], start=1):
    print(f"\n--- DIFFERENCE {i} ---")
    print("Sentence:", example["sentence"])
    print("Treebank:     ", example["treebank"])
    print("word_tokenize:", example["word_tokenize"])


--- DIFFERENCE 1 ---
Sentence: Results from No. 2 U.S. soft drink maker PepsiCo Inc . PEP.N were likely to be in the spotlight .
Treebank:      ['results', 'from', 'no.', '2', 'u.s.', 'soft', 'drink', 'maker', 'pepsico', 'inc', '.', 'pep.n', 'were', 'likely', 'to', 'be', 'in', 'the', 'spotlight', '.']
word_tokenize: ['results', 'from', 'no', '.', '2', 'u.s.', 'soft', 'drink', 'maker', 'pepsico', 'inc', '.', 'pep.n', 'were', 'likely', 'to', 'be', 'in', 'the', 'spotlight', '.']

--- DIFFERENCE 2 ---
Sentence: Also demonstrating box-office strength _ and getting seven Tony nominations _ was a potent revival of Eugene O 'Neill 's family drama , " Long Day 's Journey Into Night . "
Treebank:      ['also', 'demonstrating', 'box-office', 'strength', '_', 'and', 'getting', 'seven', 'tony', 'nominations', '_', 'was', 'a', 'potent', 'revival', 'of', 'eugene', 'o', "'neill", "'s", 'family', 'drama', ',', '``', 'long', 'day', "'s", 'journey', 'into', 'night', '.', '``']
word_tokenize: ['also', 'd

### Interpretation

Most disagreements involve abbreviations and titles such as `Gov.`, `Rev.`, `No.`, and initials such as `J.D.`. `TreebankWordTokenizer` tends to preserve these common newswire abbreviations as single tokens, while `word_tokenize` sometimes separates the abbreviation from its period.

## 7. Tokenizer Selection

Based on the comparison, `TreebankWordTokenizer` is selected as the shared tokenizer for Groups 1 and 2.

Although the two tokenizers differ on only 1.27% of the training sentences, `TreebankWordTokenizer` better preserves several common abbreviations found in MRPC newswire text. This reduces unnecessary fragmentation of their frequency counts.

In [9]:
from nltk.tokenize import TreebankWordTokenizer

tokenizer = TreebankWordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text.lower())

## 8. Training Vocabulary Analysis

After selecting the tokenizer, token frequencies are calculated using both sentences from every training pair. The vocabulary is constructed using only the training split to avoid validation or test leakage.

In [10]:
from collections import Counter

token_counts = Counter()

for column in ["sentence1", "sentence2"]:
    for sentence in train_df[column]:
        token_counts.update(tokenize(sentence))

print("Total tokens:", sum(token_counts.values()))
print("Unique tokens:", len(token_counts))

print("\n20 most common tokens:")
print(token_counts.most_common(20))

Total tokens: 161061
Unique tokens: 13905

20 most common tokens:
[('the', 8477), (',', 7978), ('.', 7510), ('to', 3613), ('of', 3420), ('in', 3119), ('a', 3109), ('and', 2879), ('``', 2523), ('said', 1796), ("'s", 1507), ('on', 1338), ('for', 1253), ('that', 1198), ('was', 1142), ('is', 916), ('at', 904), ('$', 820), ('with', 805), ('percent', 765)]


### Vocabulary Statistics

The training corpus contains 161,061 token occurrences and 13,905 unique token types before applying a minimum-frequency threshold.

The most frequent tokens are primarily punctuation and common function words such as `the`, `to`, `of`, and `in`. These tokens occur frequently but individually provide relatively little information for distinguishing paraphrases from non-paraphrases.

## 9. Minimum-Count Comparison

Several minimum-frequency thresholds are compared to determine how aggressively rare tokens should be mapped to `<UNK>`.

Increasing `min_count` reduces vocabulary size and sparsity, but it also causes more training tokens to lose their individual identities. The goal is to reduce sparsity without discarding too much information.

In [11]:
cutoffs = [1, 2, 3, 5, 10]

total_tokens = sum(token_counts.values())

for min_count in cutoffs:
    vocab = {
        token
        for token, count in token_counts.items()
        if count >= min_count
    }

    removed_types = len(token_counts) - len(vocab)

    removed_token_occurrences = sum(
        count
        for token, count in token_counts.items()
        if count < min_count
    )

    print(f"\nmin_count = {min_count}")
    print(f"Vocabulary size: {len(vocab):,}")
    print(f"Removed unique tokens: {removed_types:,}")
    print(
        f"Training token occurrences mapped to <UNK>: "
        f"{removed_token_occurrences:,} "
        f"({removed_token_occurrences / total_tokens * 100:.2f}%)"
    )


min_count = 1
Vocabulary size: 13,905
Removed unique tokens: 0
Training token occurrences mapped to <UNK>: 0 (0.00%)

min_count = 2
Vocabulary size: 9,926
Removed unique tokens: 3,979
Training token occurrences mapped to <UNK>: 3,979 (2.47%)

min_count = 3
Vocabulary size: 5,928
Removed unique tokens: 7,977
Training token occurrences mapped to <UNK>: 11,975 (7.44%)

min_count = 5
Vocabulary size: 3,753
Removed unique tokens: 10,152
Training token occurrences mapped to <UNK>: 19,565 (12.15%)

min_count = 10
Vocabulary size: 1,965
Removed unique tokens: 11,940
Training token occurrences mapped to <UNK>: 31,329 (19.45%)


### Interpretation

With `min_count = 2`, the vocabulary decreases from 13,905 to 9,926 token types. This removes 3,979 singleton token types while mapping only 2.47% of training token occurrences to `<UNK>`.

Higher thresholds remove substantially more information. For example, `min_count = 10` reduces the vocabulary to 1,965 token types and maps 19.45% of training token occurrences to `<UNK>`.

For this reason, `min_count = 2` provides a reasonable balance between reducing vocabulary sparsity and preserving information.

## 10. Final Preprocessing Decision

The shared preprocessing configuration for Groups 1 and 2 is:

- **Tokenizer:** `TreebankWordTokenizer`
- **Lowercasing:** Enabled
- **Minimum token count:** 2
- **Vocabulary size:** 9,926
- **Rare tokens:** Mapped to `<UNK>`
- **Vocabulary source:** Training split only

This configuration is kept fixed across Groups 1 and 2 so differences in model performance reflect the models themselves rather than changes in tokenization or vocabulary.